# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata properties
meta = dataset.metadata
print(f"Dataset Name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Published: {meta.datePublished}")
print(f"Version: {meta.version}")
print(f"Identifier: {meta.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant metadata defines collections of data called *record sets* (tables/files), each with its own fields (columns/attributes).

Let's print all available record set `@id`s and the associated fields and their IDs.

In [ ]:
# List all available record sets and their fields
record_sets = []
for rs in dataset.metadata.recordSets:
    print(f"RecordSet: @id = {rs['@id']}")
    record_sets.append(rs['@id'])
    fields = rs.get('field', [])
    for field in fields:
        print(f"  Field: @id = {field['@id']}  | name = {field.get('name', '')}")
    print()
print(f"All RecordSet @id values: {record_sets}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

Choose record set and field `@id`s from the overview above.

In [ ]:
# You can replace these @id values with ones listed in the overview above.
# Example: record_set_id = 'cr:RecordSet/ProteinTable'
record_sets_ids = record_sets  # using all discovered record sets
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    # List(dataset.records(...)) yields list of dicts: one per record
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    except Exception as e:
        print(f"Could not load {record_set_id}: {e}")
print(f"Loaded DataFrames: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

You can use field `@id`s (or column names as loaded) to select a numeric field and group/categorical field. Here we demonstrate with possible guesses for field IDs and will fall back to available columns if needed.

In [ ]:
# Pick a record set with data (for demonstration, use the first loaded one)
if len(dataframes):
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Analyzing first DataFrame: {record_set_id}, columns: {df.columns.tolist()}")
    # Try to find a numeric field for EDA
    import numpy as np
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Selected numeric field for analysis: {numeric_field}")
        # Filtered view
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by a likely categorical field
        candidate_group_fields = [col for col in df.columns if col not in [numeric_field] and df[col].nunique() < 10]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Average {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No dataframes loaded to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes):
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Again, use the first numeric column for visualization
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_cols[0]].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_cols[0]}")
        plt.xlabel(numeric_cols[0])
        plt.show()
        # Pairplot if at least 2 numeric columns
        if len(numeric_cols) >= 2:
            sns.pairplot(df[numeric_cols].dropna())
            plt.show()
    else:
        print("No numeric columns to visualize.")
else:
    print("No data loaded for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The FAIR<sup>2</sup> Croissant dataset was loaded and its metadata inspected using `mlcroissant`.
* The available record sets and fields were dynamically detected by their `@id`s.
* Data was extracted into Pandas DataFrames, basic filtering and normalization were demonstrated, and simple visualizations provided.
* For further analysis, consult the dataset's specific record sets and field documentation, always referencing `@id` values for accurate data extraction with `mlcroissant`.